# TA Targeted Analysis — Processing Pipeline

Processes raw targeted-analysis exports into a censored, QC-assessed dataset,
following `TA Code Spec.md`.

**Read before editing:** `docs/superpowers/specs/2026-09-15-ta-processing-design.md`
records the places where this notebook deliberately differs from the spec, and why.
The short version is that the spec was written before it was checked against real
exports, and several of its exact strings do not appear in the data.

## Status

| Spec section | State |
|---|---|
| §1 Readfile | Implemented below |
| §2 RT Validation | Not yet written |
| §3 LOQ/ULOQ | Not yet written |
| §4–§11 | Not yet written |

Sections are built and verified one at a time. Nothing below §1 exists yet.

## Setup

In [ ]:
from pathlib import Path

import pandas as pd
import yaml

# Show every compound when printing tables rather than an elided middle.
pd.set_option('display.max_rows', 200)
pd.set_option('display.width', 160)

## Sample Type vocabulary

The exports label every row with a `Sample Type`. Later sections select rows by
comparing against these values, so they are defined once here and never typed as
literal strings further down.

Two of them differ from `TA Code Spec.md`. The spec says instrument blanks are
`"Blank"` and check standards are `"Check Standard"`; the exports actually use
`"Matrix Blank"` and `"Chk Std"`. Comparing against the spec's wording would match
no rows at all — and would do so silently, producing an empty result rather than an
error. The check at the end of §1 exists to catch exactly that, now and if the
instrument software changes its wording later.

In [ ]:
# Sample Type values, exactly as they appear in the exports.
CAL_STD = 'Cal Std'
UNKNOWN = 'Unknown'
INSTRUMENT_BLANK = 'Matrix Blank'   # TA Code Spec.md calls this 'Blank'
CHECK_STANDARD = 'Chk Std'          # TA Code Spec.md calls this 'Check Standard'

KNOWN_SAMPLE_TYPES = {CAL_STD, UNKNOWN, INSTRUMENT_BLANK, CHECK_STANDARD}

# Labels that appear in Calculated Amount in place of a number.
# N/F comes from the instrument; the other two are written by §3.
NOT_FOUND = 'N/F'
BELOW_LOQ = '<LOQ'
ABOVE_ULOQ = '>ULOQ'
CENSOR_LABELS = {NOT_FOUND, BELOW_LOQ, ABOVE_ULOQ}

## §1.1 — Data folder

**To run a different batch, change `DATA_FOLDER` in the cell below.** That one line
is the only place the folder is set.

Leave it as `None` and the notebook will ask you for the folder when you run the
cell — which is what a lab member opening this for the first time will get. Either
way the answer is remembered in `answers.yaml`, so the second pass over QC-adjusted
data does not ask again and an old run can be reproduced later.

`answers.yaml` is gitignored, because the path inside it is specific to your
machine. Setting `DATA_FOLDER` below always wins over whatever is saved there.

In [ ]:
# ══════════════════════════════════════════════════════════════════════════
#  SET THE DATA FOLDER HERE  —  this is the only place it is set.
#
#  Paste the folder holding this batch's CSV exports between the quotes,
#  keeping the r before the first quote:
#
#      DATA_FOLDER = r'F:\School Folder\...\26_08_04_Oyster_RawData'
#
#  Leave it as None to be asked for the folder when you run this cell.
# ══════════════════════════════════════════════════════════════════════════

DATA_FOLDER = None

# ══════════════════════════════════════════════════════════════════════════

ANSWERS_PATH = Path('answers.yaml')
answers = yaml.safe_load(ANSWERS_PATH.read_text()) if ANSWERS_PATH.exists() else {}
answers = answers or {}

if DATA_FOLDER:
    answers['data_folder'] = str(DATA_FOLDER)
    print('Using the folder set above.')
elif answers.get('data_folder'):
    print('Using the folder remembered in answers.yaml.')
else:
    # Paths pasted from Explorer often arrive wrapped in quotes.
    answers['data_folder'] = input('Folder holding the CSV exports: ').strip().strip('"\'')
    print('Saved to answers.yaml.')

ANSWERS_PATH.write_text(yaml.safe_dump(answers, sort_keys=False))

DATA_FOLDER = Path(answers['data_folder'])
if not DATA_FOLDER.is_dir():
    raise NotADirectoryError(f'Not a folder: {DATA_FOLDER}')

print(f'Data folder: {DATA_FOLDER}')

## §1.2 — Read every export into one master table

Each export file holds exactly one compound, so the master table is the
concatenation of all of them.

**Missing values.** Columns are read as text so pandas cannot pick a different
dtype per file depending on which sentinels that file happens to contain. Note that
pandas still applies its own missing-value detection while doing so: `N/A` is on its
default list and becomes `NaN` at read time, while `N/F` is not and survives as
text. That split is what we want — `N/A` in `Theoretical Amount` means the field
does not apply to that row, whereas `N/F` is an instrument result meaning the
compound was looked for and not found, which §2 and §3 must preserve.

**Stray rows.** At least one export (`NaDONA`) ends with an extra line naming no
sample and no compound, carrying two unlabelled numbers in `Total Area` and
`ISTD Area` and nothing else. It appears to be an artifact of the export rather than
a measurement. Every real result belongs to a sample, so rows with no
`Sample Raw File Name` are dropped — and reported by file rather than removed
quietly, because a rising count here would mean something changed upstream.

`source_file` is added so any row can be traced back to the file it came from.

In [ ]:
FILE_PATTERN = '*Quantitation_ByCompound*.csv'

export_files = sorted(DATA_FOLDER.glob(FILE_PATTERN))
if not export_files:
    raise FileNotFoundError(f'No files matching {FILE_PATTERN} in {DATA_FOLDER}')

frames = []
stray_rows = []
for path in export_files:
    one_file = pd.read_csv(path, dtype=str)

    # A real measurement always names its sample. Anything else is an export
    # artifact, not data. Checked on the raw column before any renaming.
    before = len(one_file)
    one_file = one_file[one_file['Sample Raw File Name'].notna()]
    dropped = before - len(one_file)
    if dropped:
        stray_rows.append((path.name, dropped))

    one_file['source_file'] = path.name
    frames.append(one_file)

master = pd.concat(frames, ignore_index=True)

print(f'Files read: {len(export_files)}')
print(f'Rows:       {len(master):,}')
print(f'Columns:    {master.shape[1]} (before selecting the ones the spec names)')

if stray_rows:
    print(f'\nStray rows dropped ({sum(count for _, count in stray_rows)} total):')
    for name, count in stray_rows:
        print(f'  {count} from {name}')
else:
    print('\nNo stray rows found.')

## §1.3 — Keep the columns the spec names

§1.3 lists fourteen columns to record and says the rest are ignored. The exports
carry thirty.

Two of the spec's names do not match the files. `Sample Name (Batch Ordering)` is
the `Sample Name` column — the exports have a separate `Sample Order` column, and
since the spec lists `Sample ID` separately, the parenthetical is read as describing
how `Sample Name` is numbered. `ISTD Actual RT` is spelled `ISTD Actual Rt`.

Missing columns raise rather than being skipped: a renamed column upstream should
stop the run, not quietly drop data the later sections depend on.

In [ ]:
SPEC_COLUMNS = [
    'Sample Raw File Name',
    'Sample Type',
    'Sample Name',          # spec: 'Sample Name (Batch Ordering)'
    'Sample ID',
    'Compound Name',
    'Detected Mass',
    'Theoretical Amount',
    'Method Apex RT',
    'Calculated Amount',
    'Peak Area',
    'ISTD Compound Name',
    'ISTD Amount',
    'ISTD Area',
    'ISTD Actual Rt',       # spec: 'ISTD Actual RT'
]

absent = [name for name in SPEC_COLUMNS if name not in master.columns]
if absent:
    raise KeyError(f'Columns named in the spec are absent from the exports: {absent}')

master = master[SPEC_COLUMNS + ['source_file']].copy()

# Trim stray whitespace so comparisons against the constants above are reliable.
for column in master.columns:
    master[column] = master[column].str.strip()

print(f'Retained {len(SPEC_COLUMNS)} spec columns plus source_file.')

## §1.4 — Convert the numeric columns

Columns used in arithmetic are converted to numbers. Anything that cannot be parsed
— `N/F`, or `Peak index not specified` in the columns that carry it — becomes `NaN`.

**`Calculated Amount` is deliberately left as text.** §3 writes the labels `<LOQ`
and `>ULOQ` into this column, and §4.1.4 requires those labels be left in place
rather than recalculated. It therefore holds a mix of numbers and labels for the
rest of the pipeline, and every later section that does arithmetic on it must
exclude the labels explicitly and preserve them in its output.

The sentinels do **not** line up across columns, so no section may assume that a
row missing one value is missing the others. In this dataset 938 rows are `N/F` in
`Calculated Amount` but only 928 in `Method Apex RT` — ten rows have no calculated
amount yet a perfectly good retention time, which §2 has to decide what to do with.

In [ ]:
NUMERIC_COLUMNS = [
    'Detected Mass',
    'Theoretical Amount',
    'Method Apex RT',
    'Peak Area',
    'ISTD Amount',
    'ISTD Area',
    'ISTD Actual Rt',
]

for column in NUMERIC_COLUMNS:
    master[column] = pd.to_numeric(master[column], errors='coerce')

print('Converted to numeric:')
for column in NUMERIC_COLUMNS:
    parsed = master[column].notna().sum()
    print(f'  {column:22s} {parsed:>7,} of {len(master):,} values parsed')

## §1 — Checks

What to look at before moving on to §2:

- every row has a `Sample Type`, and every value is one the notebook recognises —
  either failure stops the run here rather than silently matching nothing in §7,
  §9 or §10
- the compound count matches the number of export files, confirming one compound
  per file
- `N/F` rows are present and still readable as `N/F`, not turned into `NaN`
- `Calculated Amount` is still text; every other numeric column is `float64`
- the count of compounds carrying no `ISTD Compound Name` matches the number of
  labelled standards in the method — those are the EIS/NIS compounds §7 and §8 need

In [ ]:
print('Sample Type values observed')
print(master['Sample Type'].value_counts(dropna=False).to_string())

# Counted, not dropped: a row with no Sample Type belongs to no section of the
# spec, so it must stop the run rather than be quietly skipped.
missing_type = master['Sample Type'].isna().sum()
if missing_type:
    raise ValueError(
        f'{missing_type} row(s) have no Sample Type. '
        'Check the exports for stray or partial lines.'
    )

unrecognised = set(master['Sample Type']) - KNOWN_SAMPLE_TYPES
if unrecognised:
    raise ValueError(
        f'Unrecognised Sample Type value(s): {sorted(unrecognised)}. '
        'Add them to the vocabulary cell above and check which sections they belong to.'
    )
print('\nEvery row has a recognised Sample Type.')

print(f'\nExport files:  {len(export_files)}')
print(f'Compounds:     {master["Compound Name"].nunique()}')
print(f'Samples:       {master["Sample Raw File Name"].nunique()}')
print(f'Rows:          {len(master):,}')

not_found_rows = (master['Calculated Amount'] == NOT_FOUND).sum()
print(f'\nRows with Calculated Amount = {NOT_FOUND}: {not_found_rows:,}')

# A compound with no internal standard of its own is a labelled standard.
has_istd = master.groupby('Compound Name')['ISTD Compound Name'].apply(lambda s: s.notna().any())
labelled_standards = sorted(has_istd[~has_istd].index)
print(f'Compounds with no ISTD (labelled standards): {len(labelled_standards)}')
print(f'Target compounds:                            {master["Compound Name"].nunique() - len(labelled_standards)}')

print('\nColumn dtypes')
print(master.dtypes.to_string())

In [ ]:
# First few rows, for eyeballing that the columns line up with the exports.
master.head(10)